# Who Starts Games 2 and 3? Padres Playoff Rotation

Buehler, Mize, Ray, and Pivetta since 8/20/26 vs the Cubs, Phillies, Braves, and Diamondbacks lineups post All-Star break (7/16–9/23/26).

Run top to bottom from this folder. pybaseball's cache is on, so re-runs read from disk.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pybaseball import cache, statcast, statcast_pitcher, playerid_reverse_lookup

cache.enable()
pd.set_option("display.width", 200)

FIG = "../figures"

# Padres starters: MLBAM ID, throwing hand
STARTERS = {"Buehler": (621111, "R"), "Mize": (663554, "R"), "Ray": (592662, "L"), "Pivetta": (601713, "R")}
STARTER_START, STARTER_END = "2026-08-20", "2026-09-24"

# Opponents: batting team abbreviations as Statcast uses them
TEAMS = ["CHC", "PHI", "ATL", "AZ"]
TEAM_NAME = {"CHC": "Cubs", "PHI": "Phillies", "ATL": "Braves", "AZ": "Diamondbacks"}
ASB_START = "2026-07-16"          # first game after the All-Star break
RECENT_GAMES, MIN_RECENT_PA = 15, 25

# Pitch-type buckets (knuckle curves and slurves count as curveballs)
BUCKET = {"FF": "Four-seam", "SI": "Sinker", "FC": "Cutter", "SL": "Slider", "ST": "Sweeper",
          "CU": "Curveball", "KC": "Curveball", "SV": "Curveball", "CH": "Changeup", "FS": "Splitter"}
ORDER = ["Four-seam", "Sinker", "Cutter", "Slider", "Sweeper", "Curveball", "Changeup", "Splitter"]
SWING = {"swinging_strike", "swinging_strike_blocked", "foul", "foul_tip", "hit_into_play"}
WHIFF = {"swinging_strike", "swinging_strike_blocked"}
K_EVENTS = {"strikeout", "strikeout_double_play"}

## 1. Padres starter Statcast pulls (8/20 to 9/24)

In [2]:
for name, (pid, _) in STARTERS.items():
    s = statcast_pitcher(STARTER_START, STARTER_END, pid)
    s = s.sort_values(["game_date", "at_bat_number", "pitch_number"])
    s.to_csv(f"{name.lower()}_statcast_aug20_sep24_2026.csv", index=False)
    print(f"{name:8s} {len(s):4d} pitches, {s.game_date.nunique()} games, "
          f"{s.game_date.min()} to {s.game_date.max()}")

Buehler   474 pitches, 6 games, 2026-08-23 to 2026-09-20
Mize      448 pitches, 6 games, 2026-08-22 to 2026-09-19
Ray       526 pitches, 6 games, 2026-08-24 to 2026-09-23
Pivetta   208 pitches, 3 games, 2026-09-07 to 2026-09-18


## 2. League-wide Statcast pull

Opponent batting pitches post-break, plus MLB starters since 8/20 for league-average lines.

In [3]:
# One league-wide pull (cached), split two ways:
#   opp       = every pitch where CHC/PHI/ATL/AZ are batting, post All-Star break
#   league_sp = every pitch by the 159 MLB starters (10+ IP) since 8/20, for league-average lines
mlb_sp = pd.read_csv("mlb_sp_10ip_2026.csv", encoding="utf-8-sig")
SP_IDS = set(mlb_sp["MLBAMID"])
COLS = ["game_pk", "game_date", "game_type", "home_team", "away_team", "inning_topbot", "at_bat_number",
        "pitch_number", "batter", "stand", "pitcher", "p_throws", "pitch_type", "description", "events", "type",
        "estimated_woba_using_speedangle", "woba_value", "woba_denom", "n_thruorder_pitcher"]

opp_parts, sp_parts = [], []
for s, e in [("2026-07-01", "2026-07-31"), ("2026-08-01", "2026-08-31"), ("2026-09-01", STARTER_END)]:
    raw = statcast(s, e, verbose=False)
    raw = raw[raw["game_type"] == "R"]
    bat = raw["away_team"].where(raw["inning_topbot"] == "Top", raw["home_team"])
    raw = raw.assign(bat_team=bat)
    opp_parts.append(raw[bat.isin(TEAMS) & (raw["game_date"] >= ASB_START)][COLS + ["bat_team"]])
    sp_parts.append(raw[raw["pitcher"].isin(SP_IDS) & (raw["game_date"] >= STARTER_START)][COLS])

opp = pd.concat(opp_parts, ignore_index=True)
league_sp = pd.concat(sp_parts, ignore_index=True)
opp["is_pa"] = opp["events"].notna() & (opp["events"] != "truncated_pa")
league_sp["is_pa"] = league_sp["events"].notna() & (league_sp["events"] != "truncated_pa")
for team, g in opp.groupby("bat_team"):
    print(f"{TEAM_NAME[team]:12s} {g.game_pk.nunique()} games, {len(g)} pitches, {g.is_pa.sum()} PA")
print(f"MLB SP since 8/20: {league_sp.pitcher.nunique()} pitchers, {len(league_sp)} pitches")

  0%|          | 0/31 [00:00<?, ?it/s]

  3%|▎         | 1/31 [00:00<00:11,  2.51it/s]

 42%|████▏     | 13/31 [00:00<00:00, 18.14it/s]

 71%|███████   | 22/31 [00:00<00:00, 29.59it/s]

 87%|████████▋ | 27/31 [00:01<00:00, 29.08it/s]

...

  4%|▍         | 1/24 [00:00<00:10,  2.28it/s]

 54%|█████▍    | 13/24 [00:00<00:00, 15.71it/s]

100%|██████████| 24/24 [00:01<00:00, 29.08it/s]

100%|██████████| 24/24 [00:01<00:00, 22.67it/s]

Braves       63 games, 8978 pitches, 2315 PA
Diamondbacks 62 games, 9527 pitches, 2402 PA
Cubs         62 games, 9744 pitches, 2454 PA
Phillies     61 games, 9006 pitches, 2306 PA
MLB SP since 8/20: 159 pitchers, 75006 pitches


## 3. Opponent lineups

Top 9 hitters by PA post-break with 25+ PA in the team's last 15 games.

In [4]:
# Lineup = top 9 by PA post-break, among hitters with 25+ PA in the team's last 15 games
pa = opp[opp["is_pa"]]
names = playerid_reverse_lookup(pa["batter"].unique().tolist(), key_type="mlbam")
names = names.assign(Name=names.name_first.str.title() + " " + names.name_last.str.title()).set_index("key_mlbam")["Name"]

parts = []
for team in TEAMS:
    g = pa[pa["bat_team"] == team]
    games = opp[opp["bat_team"] == team].drop_duplicates("game_pk").sort_values(["game_date", "game_pk"])
    recent = g[g["game_pk"].isin(games.tail(RECENT_GAMES)["game_pk"])].groupby("batter").size()
    agg = g.groupby("batter").agg(PA=("events", "size"),
                                  hands=("stand", lambda s: "".join(sorted(s.unique())))).sort_values("PA", ascending=False)
    agg["Bats"] = agg["hands"].map({"L": "L", "R": "R", "LR": "S"})
    agg["PA_last15"] = recent.reindex(agg.index).fillna(0).astype(int)
    top = agg[agg["PA_last15"] >= MIN_RECENT_PA].head(9)
    parts.append(top.assign(bat_team=team, Name=top.index.map(names)).reset_index())

lineups = pd.concat(parts, ignore_index=True)[["bat_team", "batter", "Name", "Bats", "PA", "PA_last15"]]
lineups.to_csv("opponent_lineups_post_asb_2026.csv", index=False)
for team in TEAMS:
    print(f"\n{TEAM_NAME[team]}")
    print(lineups[lineups.bat_team == team][["Name", "Bats", "PA", "PA_last15"]].to_string(index=False))


Cubs
               Name Bats  PA  PA_last15
Pete Crow-Armstrong    L 294         71
       Seiya Suzuki    R 278         66
      Michael Busch    L 271         61
       Alex Bregman    R 262         52
       Nico Hoerner    R 255         63
           Ian Happ    S 225         45
      Pedro Ramírez    S 191         51
       Carson Kelly    R 168         43
   Michael Conforto    L 141         45

Phillies
           Name Bats  PA  PA_last15
 Kyle Schwarber    L 269         69
   Bryce Harper    L 267         64
    Trea Turner    R 267         67
      Alec Bohm    R 239         58
   Bryson Stott    L 239         58
 J. T. Realmuto    R 197         51
  Brandon Marsh    L 194         38
    Luis Arráez    L 188         64
Justin Crawford    L 165         41

Braves
            Name Bats  PA  PA_last15
   Drake Baldwin    L 282         69
      Matt Olson    L 267         62
    Ozzie Albies    S 259         59
  Michael Harris    L 255         62
  Mauricio Dubón    R 235      

## 4. Opponent splits

xwOBA, K%, whiff% vs LHP/RHP, by pitch type, and by hitter. Whiff% = swinging strikes / swings (swings = swinging strikes, fouls, foul tips, balls in play; bunts excluded).

In [5]:
def xwoba(g):
    """Savant-style xwOBA: batted balls use estimated_woba_using_speedangle; K/BB/HBP use woba_value."""
    p = g[g["is_pa"] & (g["woba_denom"] > 0)]
    if p.empty:
        return np.nan, 0
    val = np.where(p["type"].eq("X") & p["estimated_woba_using_speedangle"].notna(),
                   p["estimated_woba_using_speedangle"], p["woba_value"])
    return val.sum() / p["woba_denom"].sum(), len(p)


def summary(g):
    x, n_pa = xwoba(g)
    p = g[g["is_pa"]]
    swings = g["description"].isin(SWING).sum()
    return pd.Series({
        "Pitches": len(g), "PA": len(p), "xwOBA": x,
        "K%": p["events"].isin(K_EVENTS).mean() if len(p) else np.nan,
        "Whiff%": g["description"].isin(WHIFF).sum() / swings if swings else np.nan,
        "xwOBA PA": n_pa,
    })


fx = lambda v: "—" if pd.isna(v) else f"{v:.3f}".lstrip("0")
fp = lambda v: "—" if pd.isna(v) else f"{v:.1%}"

# lineup hitters only
df = opp.merge(lineups[["bat_team", "batter"]], on=["bat_team", "batter"])
df["bucket"] = df["pitch_type"].map(BUCKET)

In [6]:
team_hand = {}
for team, g in df.groupby("bat_team"):
    th = g.groupby("p_throws").apply(summary).reindex(["L", "R"])
    team_hand[team] = th
    print(f"\n{TEAM_NAME[team]}: lineup vs LHP / RHP")
    print(pd.DataFrame({
        "Pitches": th["Pitches"].astype(int).values, "PA": th["PA"].astype(int).values,
        "xwOBA": th["xwOBA"].map(fx).values, "K%": th["K%"].map(fp).values, "Whiff%": th["Whiff%"].map(fp).values,
    }, index=["vs LHP", "vs RHP"]).to_string())

    print(f"\n{TEAM_NAME[team]}: xwOBA / Whiff% by pitch type (pitches, PA)")
    cells = {}
    for b in ORDER:
        row = []
        for sub in [g, g[g.p_throws == "R"], g[g.p_throws == "L"]]:
            s = summary(sub[sub.bucket == b])
            row.append(f"{fx(s['xwOBA'])} / {fp(s['Whiff%'])} ({int(s['Pitches'])}, {int(s['xwOBA PA'])} PA)")
        cells[b] = row
    print(pd.DataFrame(cells, index=["All", "vs RHP", "vs LHP"]).T.to_string())

for team, g in df.groupby("bat_team"):
    print(f"\n{TEAM_NAME[team]}: hitter xwOBA vs LHP / RHP")
    out = []
    for lu in lineups[lineups.bat_team == team].itertuples():
        h = g[g.batter == lu.batter]
        row = {"Hitter": lu.Name, "Bats": lu.Bats, "PA": lu.PA}
        for hand in ["L", "R"]:
            x, n = xwoba(h[h.p_throws == hand])
            row[f"vs {hand}HP"] = f"{fx(x)} ({n} PA)"
        out.append(row)
    print(pd.DataFrame(out).to_string(index=False))


Braves: lineup vs LHP / RHP
        Pitches    PA xwOBA     K% Whiff%
vs LHP     2211   586  .299  20.6%  22.9%
vs RHP     5433  1386  .322  22.4%  22.7%

Braves: xwOBA / Whiff% by pitch type (pitches, PA)
                                   All                       vs RHP                      vs LHP
Four-seam  .339 / 20.0% (2471, 582 PA)  .347 / 19.9% (1894, 428 PA)  .319 / 20.5% (577, 154 PA)
Sinker     .331 / 14.9% (1102, 315 PA)   .350 / 15.0% (653, 199 PA)  .298 / 14.8% (449, 116 PA)
Cutter      .306 / 20.6% (799, 188 PA)   .290 / 19.7% (544, 133 PA)   .344 / 22.4% (255, 55 PA)
Slider      .284 / 32.5% (791, 219 PA)   .300 / 34.7% (537, 144 PA)   .252 / 28.4% (254, 75 PA)
Sweeper     .291 / 29.2% (655, 168 PA)    .277 / 29.4% (418, 92 PA)   .309 / 28.8% (237, 76 PA)
Curveball   .290 / 27.1% (598, 157 PA)   .288 / 26.6% (456, 133 PA)   .305 / 29.3% (142, 24 PA)
Changeup    .302 / 22.9% (825, 213 PA)   .313 / 22.1% (599, 147 PA)   .277 / 24.8% (226, 66 PA)
Splitter    .288 / 29.7% 

                                   All                       vs RHP                      vs LHP
Four-seam  .354 / 14.3% (2577, 647 PA)  .357 / 15.3% (2115, 522 PA)  .339 / 10.0% (462, 125 PA)
Sinker      .348 / 8.4% (1249, 354 PA)    .361 / 8.9% (867, 258 PA)    .311 / 7.1% (382, 96 PA)
Cutter      .338 / 17.9% (603, 143 PA)   .362 / 17.8% (479, 111 PA)   .253 / 18.2% (124, 32 PA)
Slider     .304 / 27.9% (1109, 262 PA)   .303 / 24.9% (889, 207 PA)   .306 / 38.7% (220, 55 PA)
Sweeper     .239 / 19.1% (690, 154 PA)   .237 / 19.6% (558, 121 PA)   .250 / 17.4% (132, 33 PA)
Curveball   .337 / 27.8% (771, 172 PA)   .307 / 28.0% (612, 141 PA)   .472 / 27.1% (159, 31 PA)
Changeup   .318 / 28.9% (1003, 246 PA)   .329 / 29.2% (758, 187 PA)   .284 / 27.7% (245, 59 PA)
Splitter     .251 / 28.8% (228, 49 PA)    .251 / 28.8% (227, 49 PA)             — / — (1, 0 PA)

Phillies: lineup vs LHP / RHP
        Pitches    PA xwOBA     K% Whiff%
vs LHP     2276   592  .340  17.4%  19.8%
vs RHP     5706  1433


Braves: hitter xwOBA vs LHP / RHP
          Hitter Bats  PA       vs LHP        vs RHP
   Drake Baldwin    L 282 .274 (95 PA) .391 (186 PA)
      Matt Olson    L 267 .320 (97 PA) .361 (168 PA)
    Ozzie Albies    S 259 .277 (87 PA) .244 (172 PA)
  Michael Harris    L 255 .307 (88 PA) .308 (164 PA)
  Mauricio Dubón    R 235 .236 (71 PA) .308 (162 PA)
    Austin Riley    R 229 .346 (53 PA) .310 (175 PA)
    Ronald Acuña    R 219 .371 (62 PA) .351 (157 PA)
Mike Yastrzemski    L 130  .395 (3 PA) .319 (127 PA)
     Sean Murphy    R  96 .258 (28 PA)  .274 (68 PA)

Diamondbacks: hitter xwOBA vs LHP / RHP


         Hitter Bats  PA       vs LHP        vs RHP
Geraldo Perdomo    S 269 .297 (85 PA) .339 (179 PA)
 Gabriel Moreno    R 260 .382 (79 PA) .348 (180 PA)
 Corbin Carroll    L 256 .366 (84 PA) .339 (169 PA)
  Nolan Arenado    R 234 .307 (68 PA) .295 (166 PA)
       Tim Tawa    R 234 .309 (63 PA) .326 (167 PA)
    Ketel Marte    S 201 .298 (44 PA) .298 (153 PA)
Ildemaro Vargas    S 183 .296 (73 PA) .264 (110 PA)
  Lars Nootbaar    L 131 .384 (12 PA) .339 (116 PA)
  Jordan Lawlar    R  85 .366 (38 PA)  .240 (46 PA)

Cubs: hitter xwOBA vs LHP / RHP
             Hitter Bats  PA       vs LHP        vs RHP
Pete Crow-Armstrong    L 294 .306 (85 PA) .429 (203 PA)
       Seiya Suzuki    R 278 .410 (67 PA) .327 (210 PA)
      Michael Busch    L 271 .298 (65 PA) .318 (202 PA)
       Alex Bregman    R 262 .369 (56 PA) .338 (206 PA)
       Nico Hoerner    R 255 .256 (53 PA) .351 (201 PA)
           Ian Happ    S 225 .180 (25 PA) .323 (199 PA)
      Pedro Ramírez    S 191 .377 (34 PA) .252 (155 PA)

## 5. Matchup table

Lineup xwOBA on each pitch type (same-handed pitchers only), weighted by the starter's usage since 8/20. Diff = score minus the lineup's own xwOBA vs that hand.

In [7]:
# Matchup score = lineup xwOBA on each pitch type (same-handed pitchers only),
# weighted by the starter's usage of that pitch since 8/20. Lower = better for SD.
usage, rows = {}, []
for sp, (_, hand) in STARTERS.items():
    s = pd.read_csv(f"{sp.lower()}_statcast_aug20_sep24_2026.csv")
    u = s["pitch_type"].map(BUCKET).value_counts(normalize=True).reindex(ORDER).fillna(0)
    usage[f"{sp} ({hand}HP)"] = u
    for team, g in df.groupby("bat_team"):
        gh = g[g.p_throws == hand]
        wsum = wused = 0.0
        for b in ORDER:
            if u[b] == 0:
                continue
            x, _ = xwoba(gh[gh.bucket == b])
            if pd.isna(x):
                continue
            wsum += u[b] * x
            wused += u[b]
        weighted = wsum / wused
        base = team_hand[team].loc[hand, "xwOBA"]
        rows.append({"Starter": f"{sp} ({hand}HP)", "Opponent": TEAM_NAME[team],
                     "Weighted xwOBA": weighted, "Lineup xwOBA vs hand": base, "Diff": weighted - base})

print("Starter pitch usage since 8/20")
print(pd.DataFrame(usage).map(lambda v: f"{v:.1%}" if v else "—").to_string())

matchup = pd.DataFrame(rows)
matchup.to_csv("matchup_table_post_asb_2026.csv", index=False)
for team in ["Cubs", "Phillies", "Braves", "Diamondbacks"]:
    t = matchup[matchup.Opponent == team].sort_values("Weighted xwOBA")
    print(f"\nvs {team}")
    print(pd.DataFrame({"Starter": t["Starter"], "Weighted xwOBA": t["Weighted xwOBA"].map(fx),
                        "Lineup vs hand": t["Lineup xwOBA vs hand"].map(fx),
                        "Diff": t["Diff"].map(lambda v: f"{v:+.3f}")}).to_string(index=False))

Starter pitch usage since 8/20
           Buehler (RHP) Mize (RHP) Ray (LHP) Pivetta (RHP)
pitch_type                                                 
Four-seam          21.6%      37.1%     24.3%         46.6%
Sinker             22.9%       5.8%     22.8%          0.5%
Cutter             22.0%          —         —         18.8%
Slider             10.8%      29.7%     19.0%             —
Sweeper             7.2%          —         —          8.2%
Curveball           7.2%       3.8%     14.4%         26.0%
Changeup            8.3%          —     19.4%             —
Splitter               —      23.7%         —             —

vs Cubs
      Starter Weighted xwOBA Lineup vs hand   Diff
   Mize (RHP)           .314           .334 -0.019
    Ray (LHP)           .335           .317 +0.018
Pivetta (RHP)           .335           .334 +0.002
Buehler (RHP)           .339           .334 +0.005

vs Phillies
      Starter Weighted xwOBA Lineup vs hand   Diff
   Mize (RHP)           .316           .3

## 6. MLB SP league averages (since 8/20)

In [8]:
# MLB SP average lines for the starter chart (same 159 starters as mlb_sp_10ip_2026.csv)
lpa = league_sp[league_sp["is_pa"]]
k = lpa["events"].isin(K_EVENTS).mean()
bb = lpa["events"].isin(["walk", "intent_walk"]).mean()
ip = mlb_sp["IP"].apply(lambda v: int(v) + round((v % 1) * 10) / 3)   # FanGraphs 29.2 IP = 29 2/3
league = pd.Series({
    "K-BB%": k - bb, "BB%": bb, "xERA": (mlb_sp["xERA"] * ip).sum() / ip.sum(),
    "Stuff+": 100, "Location+": 100, "P/BF": len(league_sp) / len(lpa),
    "xwOBA 2nd TTO": xwoba(lpa[lpa["n_thruorder_pitcher"] == 2])[0],
}, name="value")
league.rename_axis("metric").to_csv("mlb_sp_league_averages_2026.csv")
print(league.round(4).to_string())

K-BB%              0.1373
BB%                0.0808
xERA               4.1773
Stuff+           100.0000
Location+        100.0000
P/BF               3.9086
xwOBA 2nd TTO      0.3093


## 7. Charts

In [9]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

BROWN, GOLD = "#5b3617", "#c9a227"          # Padres good / bad
INK, MUTED, REF, NEUTRAL = "#1f1f1f", "#6b6b6b", "#B0AFAF", "#F1EFEB"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})

In [10]:
# Chart 1: starter profile vs MLB SP average (library #81 with #72's direction rule)
lg = pd.read_csv("mlb_sp_league_averages_2026.csv").set_index("metric")["value"].to_dict()
fg = lambda f: pd.read_csv(f, encoding="utf-8-sig").set_index("MLBAMID")
adv, fsc, stf = fg("padres_sp_advanced_2026.csv"), fg("padres_sp_statcast_2026.csv"), fg("padres_sp_stuff_plus_2026.csv")


def xwoba_pa(p):
    p = p[p["woba_denom"] > 0]
    v = np.where(p["type"].eq("X") & p["estimated_woba_using_speedangle"].notna(),
                 p["estimated_woba_using_speedangle"], p["woba_value"])
    return v.sum() / p["woba_denom"].sum()


vals, bf = {}, {}
for name, (pid, _) in STARTERS.items():
    s = pd.read_csv(f"{name.lower()}_statcast_aug20_sep24_2026.csv")
    p = s[s["events"].notna() & (s["events"] != "truncated_pa")]
    bf[name] = len(p)
    vals[name] = {
        "K-BB%": adv.loc[pid, "K-BB%"], "BB%": adv.loc[pid, "BB%"], "xERA": fsc.loc[pid, "xERA"],
        "Stuff+": stf.loc[pid, "Stuff+"], "Location+": stf.loc[pid, "Location+"],
        "P/BF": len(s) / len(p), "xwOBA 2nd TTO": xwoba_pa(p[p["n_thruorder_pitcher"] == 2]),
    }

# metric: (header, higher_is_better, formatter)
pct = lambda v: f"{v:.1%}"
METRICS = {
    "K-BB%": ("K-BB%", True, pct),
    "BB%": ("BB%", False, pct),
    "xERA": ("xERA", False, lambda v: f"{v:.2f}"),
    "Stuff+": ("Stuff+", True, lambda v: f"{v:.0f}"),
    "Location+": ("Location+", True, lambda v: f"{v:.0f}"),
    "P/BF": ("Pitches / BF", False, lambda v: f"{v:.2f}"),
    "xwOBA 2nd TTO": ("xwOBA, 2nd\ntime thru order", False, lambda v: f"{v:.3f}".lstrip("0")),
}

# sign-adjusted gap to MLB SP average: positive = better
gap = {n: {m: (v[m] - lg[m]) * (1 if METRICS[m][1] else -1) for m in METRICS} for n, v in vals.items()}
order = sorted(STARTERS, key=lambda n: (sum(g > 0 for g in gap[n].values()), vals[n]["Stuff+"] + vals[n]["Location+"]))

fig, axes = plt.subplots(1, len(METRICS), figsize=(16, 6.2), sharey=True)
plt.subplots_adjust(left=0.10, right=0.97, top=0.72, bottom=0.12, wspace=0.22)
y = np.arange(len(order))
for ax, (m, (header, hib, f)) in zip(axes, METRICS.items()):
    g = np.array([gap[n][m] for n in order])
    lim = max(abs(g).max(), 1e-9) * 2.6
    ax.barh(y, g, height=0.62, color=[BROWN if v > 0 else GOLD for v in g], edgecolor="white", linewidth=2)
    ax.vlines(0, -0.6, len(order) - 0.4, color=REF, linestyle="--", linewidth=1.2)
    for yi, (gv, n) in enumerate(zip(g, order)):
        ax.text(gv + (lim * 0.06 if gv >= 0 else -lim * 0.06), yi, f(vals[n][m]), va="center",
                ha="left" if gv >= 0 else "right", fontsize=11, fontweight="bold", color=INK)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.set_xticks([])
    ax.set_title(f"{header}\n", fontsize=12, fontweight="bold", color=INK, pad=4, linespacing=1.1)
    ax.text(0.5, 1.02, "MLB SP avg " + f(lg[m]).replace("\n", " "), transform=ax.transAxes,
            ha="center", va="bottom", fontsize=9, color=MUTED)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(axis="y", length=0)
axes[0].set_yticks(y, [f"{n}\n{bf[n]} BF" for n in order], fontsize=12, fontweight="bold", color=INK)

# spanning invisible axis carries the centered title
tax = fig.add_axes([0.0, 0.925, 1.0, 0.001]); tax.axis("off")
tax.set_title("Who Starts Games 2 and 3?", fontsize=20, fontweight="bold", color=INK, pad=0)
fig.text(0.5, 0.885, "Padres SP since 8/20/26", ha="center", fontsize=13, color=MUTED)
fig.text(0.5, 0.852, "Bars right of the dashed line = better than MLB SP average; left = worse. Labels are each pitcher's actual value.",
         ha="center", fontsize=10, color=MUTED)
fig.text(0.5, 0.04, "Data: FanGraphs.com & Baseball Savant, 8/20/26–9/24/26. MLB SP avg = 159 starters with 10+ IP. "
         "Stuff+/Location+: 100 = average.", ha="center", fontsize=9, color=MUTED, style="italic")
fig.savefig(f"{FIG}/padres_game2_game3_starters_2026.png", dpi=200)
plt.close(fig)
print("Saved", f"{FIG}/padres_game2_game3_starters_2026.png")

Saved ../figures/padres_game2_game3_starters_2026.png


In [11]:
# Chart 2: best matchups by opponent (library #19 heatmap format)
mt = pd.read_csv("matchup_table_post_asb_2026.csv")
teams = ["Cubs", "Phillies", "Braves", "Diamondbacks"]
rows = ["Buehler (RHP)", "Mize (RHP)", "Ray (LHP)", "Pivetta (RHP)"]
W = mt.pivot(index="Starter", columns="Opponent", values="Weighted xwOBA").loc[rows, teams]
D = mt.pivot(index="Starter", columns="Opponent", values="Diff").loc[rows, teams]
base = mt.pivot(index="Starter", columns="Opponent", values="Lineup xwOBA vs hand").loc[rows, teams]

cmap = LinearSegmentedColormap.from_list("sd", [BROWN, NEUTRAL, GOLD])
lim = 0.03
fig = plt.figure(figsize=(10, 9.4))
GW = 0.56
ax = fig.add_axes([0.5 - GW / 2, 0.16, GW, GW * 10 / 9.4])   # grid centered on the canvas, square cells
ax.imshow(D.values, cmap=cmap, norm=TwoSlopeNorm(0, -lim, lim), aspect="equal")
fx3 = lambda v: f"{v:.3f}".lstrip("0")
for i in range(len(rows)):
    for j in range(len(teams)):
        d, w = D.iat[i, j], W.iat[i, j]
        tc = "white" if d < -lim * 0.45 else INK
        ax.text(j, i - 0.08, fx3(w), ha="center", va="center", fontsize=20, fontweight="bold", color=tc)
        ax.text(j, i + 0.26, f"{d:+.3f} vs lineup".replace("0.", ".").replace("-", "−"),
                ha="center", va="center", fontsize=9.5, color=tc)
# best (lowest weighted xwOBA) per opponent
for j, t in enumerate(teams):
    i = int(np.argmin(W[t].values))
    ax.add_patch(plt.Rectangle((j - 0.47, i - 0.47), 0.94, 0.94, fill=False, edgecolor=INK, linewidth=3))
    ax.text(j, i - 0.37, "BEST", ha="center", va="center", fontsize=8.5, fontweight="bold",
            color="white" if D.iat[i, j] < -lim * 0.45 else INK)
ax.set_xticks(range(len(teams)), [f"{t}\nvs RHP {fx3(base.loc['Buehler (RHP)', t])}\nvs LHP {fx3(base.loc['Ray (LHP)', t])}"
                                  for t in teams], fontsize=10.5, linespacing=1.3)
for lbl in ax.get_xticklabels():
    lbl.set_color(INK)
ax.xaxis.tick_top()
ax.set_yticks(range(len(rows)), rows, fontsize=13, fontweight="bold", color=INK)
ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_xticks(np.arange(-0.5, len(teams)), minor=True)
ax.set_yticks(np.arange(-0.5, len(rows)), minor=True)
ax.grid(which="minor", color="white", linewidth=4)
ax.tick_params(which="minor", length=0)
ax.set_title("Best Matchups by Opponent", fontsize=20, fontweight="bold", color=INK, pad=96)
ax.text(0.5, 1.185, "Starter pitch mix since 8/20 vs opponent lineup, post All-Star break", transform=ax.transAxes,
        ha="center", fontsize=12.5, color=MUTED)

# legend bar: brown = better for SD, gold = worse
cax = fig.add_axes([0.29, 0.088, 0.42, 0.02])
cax.imshow(np.linspace(-lim, lim, 256)[None, :], cmap=cmap, norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
cax.set_xticks([0, 127.5, 255], [f"−{lim:.3f}".replace("0.", "."), "lineup avg", f"+{lim:.3f}".replace("0.", ".")], fontsize=9)
cax.set_yticks([])
for sp in cax.spines.values():
    sp.set_visible(False)
cax.tick_params(length=0)
fig.text(0.275, 0.098, "Better for SD  ◀", ha="right", va="center", fontsize=10.5, fontweight="bold", color=BROWN)
fig.text(0.725, 0.098, "▶  Worse for SD", ha="left", va="center", fontsize=10.5, fontweight="bold", color=INK)
fig.text(0.5, 0.012, "Big number = opponent lineup's xwOBA vs each pitch type (from same-handed pitchers), weighted by the starter's usage.\n"
         "Lower = better for SD. Color = gap to that lineup's own xwOBA vs the starter's hand.\n"
         "Lineup = top 9 hitters by PA post All-Star break (7/16–9/23/26) with 25+ PA in the last 15 games. Data: Baseball Savant.",
         ha="center", fontsize=8.3, color=MUTED, style="italic", linespacing=1.45)
fig.savefig(f"{FIG}/padres_starter_matchups_2026.png", dpi=200)
plt.close(fig)
print("Saved", f"{FIG}/padres_starter_matchups_2026.png")

Saved ../figures/padres_starter_matchups_2026.png
